# EE200 Summer 2025 - Signal Processing Project
# Image Transforms, Audio Analysis, and Frequency Domain Processing

This notebook covers:
1. Basic Image Operations (resize, crop, rotate)
2. 2D Discrete Fourier Transform (DFT)
3. Frequency Domain Filtering (LPF/HPF)
4. Audio Loading and Waveform Visualization
5. Time-Frequency Analysis (Spectrogram)

In [ ]:
# Cell 1: Setup and Imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import librosa
import librosa.display
from scipy.fft import fft2, ifft2, fftshift
import os

# Set up inline plotting
%matplotlib inline
plt.style.use('default')

# Get current directory for file paths
base_path = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
print(f"Working directory: {base_path}")

---
## Part A: Image Processing

In [ ]:
# Cell 2: Load and Display Original Images
# Load grayscale images
cat_img = Image.open('cat_gray.jpg')
dog_img = Image.open('dog_gray.jpg')

# Convert to numpy arrays for processing
cat_array = np.array(cat_img)
dog_array = np.array(dog_img)

print(f"Cat image shape: {cat_array.shape}")
print(f"Dog image shape: {dog_array.shape}")
print(f"Cat dtype: {cat_array.dtype}")
print(f"Dog dtype: {dog_array.dtype}")

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Cat (Grayscale)')
axes[0].axis('off')

axes[1].imshow(dog_img, cmap='gray')
axes[1].set_title('Dog (Grayscale)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Basic Image Operations - Resize, Crop, Rotate
# Resize both images to 200x200
cat_resized = cat_img.resize((200, 200))
dog_resized = dog_img.resize((200, 200))

# Crop: (left, upper, right, lower) coordinates
cat_cropped = cat_img.crop((50, 50, 200, 200))
dog_cropped = dog_img.crop((50, 50, 200, 200))

# Rotate by 45 degrees counter-clockwise
cat_rotated = cat_img.rotate(45)
dog_rotated = dog_img.rotate(45)

# Display all operations for cat image
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cat_resized, cmap='gray')
axes[1].set_title('Resized (200x200)')
axes[1].axis('off')

axes[2].imshow(cat_cropped, cmap='gray')
axes[2].set_title('Cropped (50-200)')
axes[2].axis('off')

axes[3].imshow(cat_rotated, cmap='gray')
axes[3].set_title('Rotated (45°)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nCat image operations completed!")

In [ ]:
# Cell 4: 2D Discrete Fourier Transform (DFT)
# Compute 2D FFT for cat image
cat_fft = fft2(cat_array)
dog_fft = fft2(dog_array)

# Shift zero frequency to center
cat_fft_shift = fftshift(cat_fft)
dog_fft_shift = fftshift(dog_fft)

# Compute magnitude spectrum (log scale for better visualization)
cat_magnitude = np.log(1 + np.abs(cat_fft_shift))
dog_magnitude = np.log(1 + np.abs(dog_fft_shift))

# Compute phase spectrum
cat_phase = np.angle(cat_fft_shift)
dog_phase = np.angle(dog_fft_shift)

print(f"FFT shape: {cat_fft.shape}")
print(f"Max magnitude (log): {cat_magnitude.max():.2f}")
print(f"Phase range: [{cat_phase.min():.2f}, {cat_phase.max():.2f}] radians")

# Display magnitude spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(cat_magnitude, cmap='gray')
axes[0].set_title('Cat - Magnitude Spectrum (Log)')
axes[0].axis('off')

axes[1].imshow(dog_magnitude, cmap='gray')
axes[1].set_title('Dog - Magnitude Spectrum (Log)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Display Phase Spectra
# Display phase spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(cat_phase, cmap='twilight', aspect='auto')
axes[0].set_title('Cat - Phase Spectrum')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(dog_phase, cmap='twilight', aspect='auto')
axes[1].set_title('Dog - Phase Spectrum')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("Phase spectra show the phase angle of each frequency component.")
print("The central area contains the DC component and low frequencies.")

In [ ]:
# Cell 6: Frequency Domain Filtering - Create Filter Masks with Multiple Cutoffs
def create_ideal_filter(shape, cutoff, filter_type='low'):
    """
    Create an ideal filter in frequency domain.

    Parameters:
    - shape: tuple (M, N) image dimensions
    - cutoff: cutoff frequency D0
    - filter_type: 'low' for LPF, 'high' for HPF

    Returns:
    - filter mask of same shape
    """
    M, N = shape
    # Create coordinate grids centered at zero
    u = np.arange(-M//2, M//2)
    v = np.arange(-N//2, N//2)
    V, U = np.meshgrid(v, u)

    # Compute distance from center (frequency radius)
    D = np.sqrt(U**2 + V**2)

    if filter_type == 'low':
        # Ideal Low-Pass Filter: 1 inside radius, 0 outside
        H = (D <= cutoff).astype(float)
    else:  # high-pass
        # Ideal High-Pass Filter: 0 inside radius, 1 outside
        H = (D > cutoff).astype(float)

    return H

# Get image dimensions
M, N = cat_array.shape

# Create filters at THREE different cutoffs for visible effects
cutoff_conservative = min(M, N) // 4   # 25% - subtle blur
cutoff_default = min(M, N) // 8         # 12.5% - moderate effect
cutoff_aggressive = min(M, N) // 16     # 6.25% - strong blur

print(f"Image dimensions: {M} x {N}")
print(f"Conservative cutoff (25%): {cutoff_conservative} pixels - subtle blur")
print(f"Default cutoff (12.5%): {cutoff_default} pixels - moderate effect")
print(f"Aggressive cutoff (6.25%): {cutoff_aggressive} pixels - strong blur")

# Create LPF masks at all cutoffs
lpf_conservative = create_ideal_filter((M, N), cutoff_conservative, 'low')
lpf_default = create_ideal_filter((M, N), cutoff_default, 'low')
lpf_aggressive = create_ideal_filter((M, N), cutoff_aggressive, 'low')

# Create HPF mask (edges look similar at different cutoffs)
hpf_default = create_ideal_filter((M, N), cutoff_default, 'high')

# Visualize all filter masks for comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(lpf_conservative, cmap='gray')
axes[0].set_title(f'LPF Conservative\n(D0={cutoff_conservative}, 25%)')
axes[0].axis('off')

axes[1].imshow(lpf_default, cmap='gray')
axes[1].set_title(f'LPF Default\n(D0={cutoff_default}, 12.5%)')
axes[1].axis('off')

axes[2].imshow(lpf_aggressive, cmap='gray')
axes[2].set_title(f'LPF Aggressive\n(D0={cutoff_aggressive}, 6.25%)')
axes[2].axis('off')

axes[3].imshow(hpf_default, cmap='gray')
axes[3].set_title(f'HPF Default\n(D0={cutoff_default})')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nFilter masks show which frequencies are passed (white = 1) or blocked (black = 0)")

In [ ]:
# Cell 7: Apply Filters with Multiple Cutoff Comparison
# Apply LPF at different cutoffs to cat image
cat_lpf_conservative = np.real(ifft2(fftshift(cat_fft_shift * lpf_conservative)))
cat_lpf_default = np.real(ifft2(fftshift(cat_fft_shift * lpf_default)))
cat_lpf_aggressive = np.real(ifft2(fftshift(cat_fft_shift * lpf_aggressive)))

# Apply HPF to cat image
cat_hpf = np.real(ifft2(fftshift(cat_fft_shift * hpf_default)))

# Apply LPF at default cutoff to dog image
dog_lpf_default = np.real(ifft2(fftshift(dog_fft_shift * lpf_default)))

# Apply HPF to dog image
dog_hpf = np.real(ifft2(fftshift(dog_fft_shift * hpf_default)))

print("Filtering completed successfully!")
print(f"\nCat LPF results:")
print(f"  Conservative (25%): range [{cat_lpf_conservative.min():.2f}, {cat_lpf_conservative.max():.2f}]")
print(f"  Default (12.5%): range [{cat_lpf_default.min():.2f}, {cat_lpf_default.max():.2f}]")
print(f"  Aggressive (6.25%): range [{cat_lpf_aggressive.min():.2f}, {cat_lpf_aggressive.max():.2f}]")
print(f"\nCat HPF result:")
print(f"  Default (12.5%): range [{cat_hpf.min():.2f}, {cat_hpf.max():.2f}]")

In [ ]:
# Cell 8b: Rotate Image and Compare 2D DFT Spectra
# Rotate the cat image anti-clockwise 90 degrees
cat_rotated_90 = cat_img.rotate(90)

# Convert to numpy array
cat_rotated_array = np.array(cat_rotated_90)

# Compute 2D DFT of rotated image
cat_rotated_fft = fft2(cat_rotated_array)
cat_rotated_fft_shift = fftshift(cat_rotated_fft)

# Compute magnitude spectrum (log scale)
cat_rotated_magnitude = np.log(1 + np.abs(cat_rotated_fft_shift))

print("=" * 70)
print("ROTATED IMAGE ANALYSIS - 90° Anti-Clockwise Rotation")
print("=" * 70)
print(f"Original shape: {cat_array.shape}")
print(f"Rotated shape: {cat_rotated_array.shape}")

# Display comparison: Original, Rotated, Original FFT, Rotated FFT
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original image
axes[0, 0].imshow(cat_array, cmap='gray')
axes[0, 0].set_title('Original Cat Image')
axes[0, 0].axis('off')

# Rotated image
axes[0, 1].imshow(cat_rotated_array, cmap='gray')
axes[0, 1].set_title('Rotated (90° Anti-Clockwise)')
axes[0, 1].axis('off')

# Original magnitude spectrum
axes[1, 0].imshow(cat_magnitude, cmap='gray')
axes[1, 0].set_title('Original Magnitude Spectrum')
axes[1, 0].axis('off')

# Rotated magnitude spectrum
axes[1, 1].imshow(cat_rotated_magnitude, cmap='gray')
axes[1, 1].set_title('Rotated Magnitude Spectrum')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("OBSERVATIONS - Comparing Original vs Rotated FFT Spectra")
print("=" * 70)
print("""
1. ROTATION EFFECT: When an image is rotated by θ, its Fourier transform
   is also rotated by the same angle θ in the frequency domain.

2. SPECTRAL PATTERN: The rotated FFT shows the same energy distribution
   as the original, but rotated by 90° to match the spatial rotation.

3. MAGNITUDE RELATIONSHIP: |F(u,v)| = |F_rotated(u,v)| after rotation
   - The magnitude of the Fourier transform is rotation-invariant
   - Only the orientation changes, not the magnitude distribution

4. ENERGY CONSERVATION: Both spectra have the same total energy
   - Sum of squared magnitudes is preserved under rotation
   - This demonstrates Parseval's theorem in 2D

5. PRACTICAL IMPLICATION: FFT-based processing (filtering, etc.)
   works the same way regardless of image orientation.
""")

---
## Part B: Audio Signal Processing

In [ ]:
# Cell 9b: Restore Audio - Audio Signal Processing
# Apply audio filters and reconstruction

print("=" * 70)
print("AUDIO RESTORATION - Frequency Domain Filtering for Audio")
print("=" * 70)

# Apply a low-pass filter to the audio to remove high frequencies (noise)
# This demonstrates that LPF/HPF concepts work the same for audio as for images

# Parameters for audio filtering
audio_D0 = 1000  # Cutoff frequency in Hz

print(f"""
AUDIO FILTERING PARAMETERS:
==========================
Original sampling rate: {sr} Hz
Nyquist frequency: {sr/2} Hz
LPF Cutoff frequency: {audio_D0} Hz

Note: Unlike images where we filter 2D frequency domain,
for audio we filter the 1D frequency spectrum.
""")

# For audio, we use a simple approach: apply FFT, filter, and reconstruct
# This demonstrates the same principle as image filtering

# Compute FFT of audio signal
y_fft = np.fft.rfft(y)
frequencies_audio = np.fft.rfftfreq(len(y), 1/sr)

# Create audio filter mask
audio_lpf_mask = (np.abs(frequencies_audio) <= audio_D0).astype(float)
audio_hpf_mask = (np.abs(frequencies_audio) > audio_D0).astype(float)

# Apply filters
y_lpf = np.fft.irfft(y_fft * audio_lpf_mask)
y_hpf = np.fft.irfft(y_fft * audio_hpf_mask)

print(f"✓ Applied LPF with cutoff {audio_D0} Hz")
print(f"✓ Applied HPF with cutoff {audio_D0} Hz")

# Compare original and filtered audio
print("\n--- Audio Restoration Comparison ---")
print(f"Original audio range: [{y.min():.4f}, {y.max():.4f}]")
print(f"LPF filtered range: [{y_lpf.min():.4f}, {y_lpf.max():.4f}]")
print(f"HPF filtered range: [{y_hpf.min():.4f}, {y_hpf.max():.4f}]")

# Note: Full audio playback would require saving to file
print("""
NOTE: For full audio restoration playback:
- Save filtered audio using librosa.output.write_wav()
- Compare the sound quality of original vs filtered audio
- LPF removes hiss/noise (high frequencies)
- HPF removes rumble/bass (low frequencies)
""")

print("\n" + "=" * 70)
print("AUDIO RESTORATION COMPLETE")
print("=" * 70)

In [ ]:
# Cell 10: Normalize and Display Waveform
# Normalize audio to [-1, 1] range
y_normalized = y / np.max(np.abs(y))

print(f"Normalized range: [{y_normalized.min():.4f}, {y_normalized.max():.4f}]")

# Plot waveform
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_normalized, sr=sr)
plt.title("Audio Waveform - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nWaveform shows amplitude variation over time.")
print("The piccolo melody creates distinct amplitude patterns.")

In [ ]:
# Cell 11: STFT and Spectrogram
# Compute Short-Time Fourier Transform (STFT)
D = librosa.stft(y)

print(f"STFT shape: {D.shape}")
print(f"STFT dtype: {D.dtype}")

# Convert to dB scale
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

print(f"dB range: [{S_db.min():.2f}, {S_db.max():.2f}] dB")

# Plot spectrogram
plt.figure(figsize=(12, 5))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title("Spectrogram - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

print("\nSpectrogram shows frequency content over time.")
print("Brighter colors = higher energy at that frequency.")

In [ ]:
# Cell 12: Spectral Analysis - Identify Dominant Frequencies
# Compute mean spectrum across time
mean_spectrum = np.mean(np.abs(D), axis=1)
frequencies = librosa.fft_frequencies(sr=sr)

# Plot power spectral density
plt.figure(figsize=(12, 4))
plt.semilogy(frequencies, mean_spectrum)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Mean Magnitude')
plt.title('Average Power Spectrum')
plt.grid(True, alpha=0.3)
plt.xlim([0, sr/2])
plt.tight_layout()
plt.show()

# Find peak frequencies (dominant harmonics)
from scipy.signal import find_peaks

peaks, properties = find_peaks(mean_spectrum, height=np.max(mean_spectrum)*0.1)
peak_frequencies = frequencies[peaks]
peak_magnitudes = mean_spectrum[peaks]

# Sort by magnitude and show top 10
sorted_idx = np.argsort(peak_magnitudes)[::-1][:10]
print("\nTop 10 Dominant Frequencies:")
print("-" * 30)
for i, idx in enumerate(sorted_idx):
    print(f"{i+1}. {peak_frequencies[idx]:.1f} Hz (magnitude: {peak_magnitudes[idx]:.4f})")

print("\nPiccolo frequencies typically range from 500 Hz to 4 kHz.")

In [ ]:
# Cell 13: Frequency Mixer - Creative Image Fusion
# This system fuses two images: one provides fine details, the other provides structure

print("=" * 70)
print("FREQUENCY MIXER - Creative Image Fusion System")
print("=" * 70)
print("""
SYSTEM DESIGN: Frequency Mixer
===============================
A frequency mixer combines the low-frequency (structure) information from
one image with the high-frequency (fine details) information from another.

This creates a hybrid image where different perceptual information comes
from different source images.

TRANSFER FUNCTION:
-----------------
H_mixer(u,v) = H_LPF(u,v) for Image A (structure) + H_HPF(u,v) for Image B (details)

Where:
- H_LPF = 1 for |F| <= D0, 0 otherwise (low frequencies pass)
- H_HPF = 1 for |F| > D0, 0 otherwise (high frequencies pass)
""")

# Define cutoff frequency for mixing
# Use 10% of the smaller dimension for good separation
D0 = min(cat_array.shape) // 10  # Cutoff radius

print(f"\nCutoff Frequency (D0): {D0} pixels")

# Create ideal bandpass filters for the mixer
def create_band_filters(shape, cutoff):
    """
    Create LPF and HPF for frequency mixing.
    
    Returns:
    - LPF mask: passes low frequencies (structure)
    - HPF mask: passes high frequencies (details)
    """
    M, N = shape
    u = np.arange(-M//2, M//2)
    v = np.arange(-N//2, N//2)
    V, U = np.meshgrid(v, u)
    D = np.sqrt(U**2 + V**2)
    
    lpf = (D <= cutoff).astype(float)  # Structure mask
    hpf = (D > cutoff).astype(float)   # Details mask
    
    return lpf, hpf

# Create filters
lpf_mixer, hpf_mixer = create_band_filters(cat_array.shape, D0)

# Compute FFT of both images
cat_fft_shift = fftshift(fft2(cat_array))
dog_fft_shift = fftshift(fft2(dog_array))

# Apply frequency mixer
# Option 1: Cat provides structure, Dog provides details
# Option 2: Dog provides structure, Cat provides details

# Mix 1: Cat (structure) + Dog (details)
mixed_fft_1 = (cat_fft_shift * lpf_mixer) + (dog_fft_shift * hpf_mixer)
mixed_image_1 = np.real(ifft2(fftshift(mixed_fft_1)))

# Mix 2: Dog (structure) + Cat (details)
mixed_fft_2 = (dog_fft_shift * lpf_mixer) + (cat_fft_shift * hpf_mixer)
mixed_image_2 = np.real(ifft2(fftshift(mixed_fft_2)))

# Display the frequency mixer system
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1: Mix 1 (Cat structure + Dog details)
axes[0, 0].imshow(cat_array, cmap='gray')
axes[0, 0].set_title('Image A: Cat\n(provides structure)')
axes[0, 0].axis('off')

axes[0, 1].imshow(dog_array, cmap='gray')
axes[0, 1].set_title('Image B: Dog\n(provides details)')
axes[0, 1].axis('off')

axes[0, 2].imshow(lpf_mixer, cmap='gray')
axes[0, 2].set_title(f'LPF Transfer Function\n(D0={D0})')
axes[0, 2].axis('off')

axes[0, 3].imshow(np.clip(mixed_image_1, 0, 255), cmap='gray')
axes[0, 3].set_title('Mixed: Cat structure\n+ Dog details')
axes[0, 3].axis('off')

# Row 2: Mix 2 (Dog structure + Cat details)
axes[1, 0].imshow(dog_array, cmap='gray')
axes[1, 0].set_title('Image A: Dog\n(provides structure)')
axes[1, 0].axis('off')

axes[1, 1].imshow(cat_array, cmap='gray')
axes[1, 1].set_title('Image B: Cat\n(provides details)')
axes[1, 1].axis('off')

axes[1, 2].imshow(hpf_mixer, cmap='gray')
axes[1, 2].set_title('HPF Transfer Function\n(D0={})'.format(D0))
axes[1, 2].axis('off')

axes[1, 3].imshow(np.clip(mixed_image_2, 0, 255), cmap='gray')
axes[1, 3].set_title('Mixed: Dog structure\n+ Cat details')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

# Plot the 2D transfer functions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# LPF transfer function as surface
ax1 = fig.add_subplot(131, projection='3d')
u = np.arange(-cat_array.shape[0]//2, cat_array.shape[0]//2)
v = np.arange(-cat_array.shape[1]//2, cat_array.shape[1]//2)
V, U = np.meshgrid(v, u)
D = np.sqrt(U**2 + V**2)
Z_lpf = (D <= D0).astype(float)
ax1.plot_surface(V, U, Z_lpf, cmap='gray', alpha=0.8)
ax1.set_title('LPF Transfer Function\n(Structure Mask)')
ax1.set_xlabel('u (frequency)')
ax1.set_ylabel('v (frequency)')
ax1.set_zlabel('H(u,v)')

# HPF transfer function as surface
ax2 = fig.add_subplot(132, projection='3d')
Z_hpf = (D > D0).astype(float)
ax2.plot_surface(V, U, Z_hpf, cmap='gray', alpha=0.8)
ax2.set_title('HPF Transfer Function\n(Details Mask)')
ax2.set_xlabel('u (frequency)')
ax2.set_ylabel('v (frequency)')
ax2.set_zlabel('H(u,v)')

# Combined mixer transfer function
ax3 = fig.add_subplot(133, projection='3d')
Z_combined = np.zeros_like(Z_lpf)
for i in range(cat_array.shape[0]):
    for j in range(cat_array.shape[1]):
        if D[i, j] <= D0:
            Z_combined[i, j] = 1  # LPF for Image A
        else:
            Z_combined[i, j] = 2  # HPF for Image B

ax3.plot_surface(V, U, Z_combined, cmap='coolwarm', alpha=0.8)
ax3.set_title('Mixer Transfer Function\n(1=Structure, 2=Details)')
ax3.set_xlabel('u (frequency)')
ax3.set_ylabel('v (frequency)')
ax3.set_zlabel('Source')

# Remove the axes from 2D subplot
axes[0].axis('off')
axes[1].axis('off')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("FREQUENCY MIXER - Transfer Functions")
print("=" * 70)
print(f"""
2D TRANSFER FUNCTION PLOTS:
==========================

H_LPF(u,v) = 1  if sqrt(u² + v²) <= D0
           = 0  otherwise

H_HPF(u,v) = 0  if sqrt(u² + v²) <= D0
           = 1  otherwise

Where D0 (cutoff) = {D0} pixels

H_MIXER(u,v) = H_LPF(u,v) × Image_A(u,v) + H_HPF(u,v) × Image_B(u,v)

""")

print("=" * 70)
print("OBSERVATIONS - Frequency Mixer Results")
print("=" * 70)
print("""
1. HYBRID IMAGE CREATION: The mixer successfully creates hybrid images
   combining structure from one image with details from another.

2. STRUCTURE vs DETAILS:
   - Low frequencies (center of FFT) control overall shape and structure
   - High frequencies (edges of FFT) control fine details and textures

3. PERCEPTUAL ANALYSIS:
   - Mixed 1 (Cat structure + Dog details): Cat outline, dog texture
   - Mixed 2 (Dog structure + Cat details): Dog outline, cat texture

4. CUTOFF DEPENDENCE:
   - Smaller D0: More structure from Image A, more details from Image B
   - Larger D0: Less distinction between sources

5. APPLICATION: This technique is used in:
   - Image blending and composition
   - Artistic effects
   - Medical imaging fusion
   - Multi-focus image combination
""")

In [ ]:
# Cell 14: Summary and Conclusions
print("=" * 70)
print("EE200 Signal Processing Project - COMPLETE Summary")
print("=" * 70)

print("\n" + "-" * 70)
print("PART A: IMAGE PROCESSING")
print("-" * 70)
print(f"✓ Loaded grayscale images: cat ({cat_array.shape}), dog ({dog_array.shape})")
print(f"✓ Performed basic operations: resize, crop, rotate")
print(f"✓ Computed 2D DFT and displayed magnitude/phase spectra")
print(f"✓ Designed ideal LPF at 3 cutoffs (6.25%, 12.5%, 25%)")
print(f"✓ Designed ideal HPF for edge enhancement")
print(f"✓ Applied frequency domain filtering - visible blur and edge effects")
print(f"✓ Rotated image 90° anti-clockwise and compared FFT spectra")
print(f"✓ Implemented Frequency Mixer for creative image fusion")

print("\n" + "-" * 70)
print("PART B: AUDIO PROCESSING")
print("-" * 70)
print(f"✓ Loaded audio: 'song_with_2piccolo.wav'")
print(f"✓ Sampling rate: {sr} Hz, Duration: {duration:.2f}s")
print(f"✓ Plotted normalized waveform")
print(f"✓ Computed STFT and dB spectrogram")
print(f"✓ Identified dominant frequencies in piccolo range (500Hz-4kHz)")
print(f"✓ Applied audio frequency domain filtering (LPF/HPF)")

print("\n" + "=" * 70)
print("KEY CONCEPTS DEMONSTRATED")
print("=" * 70)
print("""
1. 2D DISCRETE FOURIER TRANSFORM (DFT):
   - Transforms image from spatial to frequency domain
   - Magnitude spectrum shows frequency content
   - Phase spectrum contains spatial relationship information
   - Rotation property: FFT of rotated image is rotated FFT

2. FREQUENCY DOMAIN FILTERING:
   - LPF (Low-Pass Filter): Removes high frequencies → Blur effect
   - HPF (High-Pass Filter): Removes low frequencies → Edge enhancement
   - Multiple cutoff values demonstrate varying blur levels

3. FREQUENCY MIXER (Image Fusion):
   - Combines structure from one image with details from another
   - H_MIXER = H_LPF × Image_A + H_HPF × Image_B
   - Creates hybrid images with mixed perceptual information

4. AUDIO SIGNAL PROCESSING:
   - STFT: Time-frequency analysis for audio
   - Spectrogram: Visual frequency content over time
   - FFT-based filtering for audio restoration
""")

print("\n" + "=" * 70)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("=" * 70)